In [ ]:
#values for c2, c3, t3-s2, t3-s3, t3,s2-r4 where obtained from empirical observation, see more at imt_data.ipynb 


import numpy as np

vehicle_catalog = {
    "M": {
        "n_axles": 1,
        "Pi": lambda: np.random.normal([150.0], 0.10 * np.array([150.0])),
        "di": np.array([]),
        "notes": "motorcycle (single axle, parametric, not weighed by IMT)"
    },
    "A": {
        "n_axles": 2,
        "Pi": lambda: np.random.normal([560.0, 840.0], 0.10 * np.array([560.0, 840.0])),
        "di": np.array([2.7]),
        "notes": "passenger car (2-axle, parametric, not weighed by IMT)"
    },
    "B": {
        "n_axles": 2,
        "Pi": lambda: np.random.normal([7000.0, 7000.0], 0.10 * np.array([7000.0, 7000.0])),
        "di": np.array([4.0]),
        "notes": "bus (2-axle, parametric, not weighed by IMT)"
    },
    "C2": {
        "n_axles": 2,
        "Pi": lambda: np.random.normal(
            [2814.0, 4402.0],
            [1371.0, 2147.0]
        ),
        "di": np.array([3.5]),
        "notes": "C2 (IMT empirical, n=1063, PBV mean 7.2 t, max 44.7 t)"
    },
    "C3": {
        "n_axles": 3,
        "Pi": lambda: np.random.normal(
            [3460.0, 4382.0, 4510.0],
            [1770.0, 2229.0, 2304.0]
        ),
        "di": np.array([3.5, 1.5]),
        "notes": "C3 (IMT empirical, n=994, PBV mean 12.4 t, max 45.8 t)"
    },
    "T3S2": {
        "n_axles": 5,
        "Pi": lambda: np.random.normal(
            [2690.0, 5145.0, 5387.0, 5149.0, 5556.0],
            [851.0, 1624.0, 1699.0, 1627.0, 1782.0]
        ),
        "di": np.array([1.4, 1.4, 6.0, 1.3]),
        "notes": "T3-S2 (IMT empirical, n=1327, PBV mean 23.9 t, max 61.0 t)"
    },
    "T3S3": {
        "n_axles": 6,
        "Pi": lambda: np.random.normal(
            [3294.0, 5102.0, 5272.0, 5236.0, 5420.0, 5451.0],
            [1010.0, 1557.0, 1605.0, 1607.0, 1692.0, 1711.0]
        ),
        "di": np.array([1.4, 1.4, 6.0, 1.3, 1.3]),
        "notes": "T3-S3 (IMT empirical, n=295, PBV mean 29.8 t, max 63.6 t)"
    },
    "T3S2R4": {
        "n_axles": 9,
        "Pi": lambda: np.random.normal(
            [2789.0, 6854.0, 6863.0, 6510.0, 6516.0, 6500.0, 6503.0, 6445.0, 7130.0],
            [1077.0, 2574.0, 2556.0, 2448.0, 2436.0, 2433.0, 2440.0, 2404.0, 2809.0]
        ),
        "di": np.array([1.4, 1.4, 6.0, 1.3, 6.0, 1.3, 1.3, 1.3]),
        "notes": "T3-S2-R4 (IMT empirical, n=704, PBV mean 56.1 t, max 92.4 t, heaviest legal Mex)"
    },
    "Otros": {
        "n_axles": 2,
        "Pi": lambda: np.random.normal([4000.0, 8000.0], 0.10 * np.array([4000.0, 8000.0])),
        "di": np.array([3.5]),
        "notes": "default light truck (parametric)"
    }
}


def sample_vehicle(class_probs, catalog=vehicle_catalog, rng=None):
    """
    This function samples a single vehicle class according to class_probs and return a vehicle dict
    compatible with latter moment calculations.
    """
    if rng is None:
        rng = np.random.default_rng()

    available = [k for k in class_probs.keys() if k in catalog]
    if len(available) == 0:
        raise ValueError("No overlapping classes between provided probabilities and catalog.")

    probs = np.array([class_probs[k] for k in available], dtype=float)
    if probs.sum() <= 0:
        raise ValueError("Probabilities must sum to a positive number.")
    probs = probs / probs.sum()

    choice = rng.choice(available, p=probs)
    template = catalog[choice]

    if callable(template["Pi"]):
        Pi = np.array(template["Pi"](), dtype=float)
    else:
        Pi = np.array(template["Pi"], dtype=float)

    # disallows negative values of Pi. 
    Pi = np.maximum(Pi, 0.0)

    di = np.array(template["di"], dtype=float) if "di" in template else np.array([])

    vehicle = {
        "class": choice,
        "Pi": Pi,
        "di": di,
        "n_axles": int(template["n_axles"]),
        "prob": float(class_probs[choice]) / (sum(class_probs.values()) if sum(class_probs.values()) != 0 else 1.0),
        "notes": template.get("notes", "")
    }
    return vehicle
